In [5]:
"""
Warehouse Delivery Robot - Search Algorithms (Intelligent Systems, Task 1)
===========================================================================
Algorithms implemented from scratch (no path-finding libraries):
  1. UCS  - uninformed, minimises total cost (Dijkstra-style)
  2. A*   - informed, f(n) = g(n) + h(n), h = Manhattan distance

HOW TO RUN
----------
  python warehouse_robot.py          -> runs all algorithms on MAP_A, MAP_B, MAP_C
                                        (and prints the summary table)
HOW TO CHANGE THE MAP
---------------------
  * Edit the variable GRID right below (e.g. GRID = MAP_B), then run the file
    with the flag --single to run only on GRID:
        python warehouse_robot.py --single
  * Or write your own map as a list of strings and assign it to GRID.
"""
import heapq
import sys
import time

# --------------------------------------------------------------------------
# Maps  (S start, G goal, . cost 1, ~ cost 3, # wall)
# --------------------------------------------------------------------------
MAP_A = [
    "S..#....",
    "##.#.##.",
    "....~~..",
    ".####.#.",
    "...~..#G",
]
MAP_B = [
    "S~~~~~~~G",
    ".#######.",
    ".........",
    ".#.#.#.#.",
    ".........",
]
MAP_C = [
    "S..#....",
    ".#.#.##.",
    "...#..#.",
    ".###.##.",
    "....#..G",
]

GRID = MAP_A          # <<< change this to MAP_B / MAP_C / your own map

MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # Up, Down, Left, Right


# --------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------
def find_cell(grid, ch):
    """Scan the grid and return (row, col) of the first cell equal to ch."""
    for r, row in enumerate(grid):
        for c, cell in enumerate(row):
            if cell == ch:
                return (r, c)
    return None


def step_cost(grid, cell):
    """Cost of ENTERING a cell: '~' = 3, everything else walkable = 1."""
    return 3 if grid[cell[0]][cell[1]] == "~" else 1


def neighbors(grid, cell):
    """Yield walkable neighbours (Up, Down, Left, Right) that are not walls."""
    r, c = cell
    for dr, dc in MOVES:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[nr]) and grid[nr][nc] != "#":
            yield (nr, nc)


def manhattan(a, b):
    """Heuristic: Manhattan distance. Admissible & consistent because the
    cheapest possible move costs 1 and the robot moves 4-directionally."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def rebuild_path(parent, goal):
    path, node = [], goal
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]


def path_cost(grid, path):
    """Sum of entering costs (start cell is not counted)."""
    return sum(step_cost(grid, p) for p in path[1:])


def result(path, grid, expanded, max_frontier, elapsed):
    return {"path": path,
            "cost": path_cost(grid, path) if path else None,
            "expanded": expanded,
            "max_frontier": max_frontier,
            "time_ms": elapsed * 1000}


# --------------------------------------------------------------------------
# 1) UCS  (uninformed - cheapest path; expands by lowest g(n))
# --------------------------------------------------------------------------
def ucs(grid, start, goal):
    t0 = time.perf_counter()
    frontier = [(0, start)]               # priority queue of (g, cell)
    best_g = {start: 0}
    parent = {start: None}
    closed = set()
    expanded, max_f = 0, 1
    while frontier:
        g, node = heapq.heappop(frontier)
        if node in closed:                # stale queue entry
            continue
        closed.add(node)
        expanded += 1
        if node == goal:
            return result(rebuild_path(parent, goal), grid, expanded, max_f,
                          time.perf_counter() - t0)
        for nb in neighbors(grid, node):
            ng = g + step_cost(grid, nb)
            if nb not in best_g or ng < best_g[nb]:
                best_g[nb] = ng
                parent[nb] = node
                heapq.heappush(frontier, (ng, nb))
        max_f = max(max_f, len(frontier))
    return result(None, grid, expanded, max_f, time.perf_counter() - t0)


# --------------------------------------------------------------------------
# 2) A*  (informed - f = g + h, h = Manhattan distance)
# --------------------------------------------------------------------------
def astar(grid, start, goal):
    t0 = time.perf_counter()
    frontier = [(manhattan(start, goal), 0, start)]   # (f, g, cell)
    best_g = {start: 0}
    parent = {start: None}
    closed = set()
    expanded, max_f = 0, 1
    while frontier:
        f, g, node = heapq.heappop(frontier)
        if node in closed:
            continue
        closed.add(node)
        expanded += 1
        if node == goal:
            return result(rebuild_path(parent, goal), grid, expanded, max_f,
                          time.perf_counter() - t0)
        for nb in neighbors(grid, node):
            ng = g + step_cost(grid, nb)
            if nb not in best_g or ng < best_g[nb]:
                best_g[nb] = ng
                parent[nb] = node
                heapq.heappush(frontier, (ng + manhattan(nb, goal), ng, nb))
        max_f = max(max_f, len(frontier))
    return result(None, grid, expanded, max_f, time.perf_counter() - t0)


# --------------------------------------------------------------------------
# Display
# --------------------------------------------------------------------------
def draw(grid, path):
    """Return the map as text with path cells marked '*' (S and G kept)."""
    cells = set(path or [])
    lines = []
    for r, row in enumerate(grid):
        lines.append("".join("*" if (r, c) in cells and ch == "." or
                             (r, c) in cells and ch == "~" else ch
                             for c, ch in enumerate(row)))
    return "\n".join(lines)


ALGORITHMS = [("UCS", ucs), ("A* (Manhattan)", astar)]


def run_map(name, grid, table):
    start, goal = find_cell(grid, "S"), find_cell(grid, "G")
    print("=" * 60)
    print(f"{name}   start={start}  goal={goal}")
    print("=" * 60)
    for algo_name, fn in ALGORITHMS:
        res = fn(grid, start, goal)
        print(f"\n--- {algo_name} ---")
        if res["path"] is None:
            print("No path found")
            print(f"Nodes expanded : {res['expanded']}")
        else:
            print("Path           :", res["path"])
            print("Path length    :", len(res["path"]) - 1, "moves")
            print("Total cost     :", res["cost"])
            print("Nodes expanded :", res["expanded"])
            print("Map (* = path):")
            print(draw(grid, res["path"]))
        table.append((name, algo_name, res))
    print()


def print_table(table):
    print("=" * 88)
    print(f"{'Map':6}{'Algorithm':16}{'Length':>8}{'Cost':>9}{'Expanded':>10}"
          f"{'MaxFrontier':>13}{'Time(ms)':>10}")
    print("-" * 88)
    for name, algo, r in table:
        length = len(r["path"]) - 1 if r["path"] else "-"
        cost = r["cost"] if r["path"] else "No path"
        print(f"{name:6}{algo:16}{str(length):>8}{str(cost):>9}{r['expanded']:>10}"
              f"{r['max_frontier']:>13}{r['time_ms']:>10.3f}")
    print("=" * 88)


if __name__ == "__main__":
    table = []
    if "--single" in sys.argv:
        run_map("GRID", GRID, table)
    else:
        for n, g in (("MAP_A", MAP_A), ("MAP_B", MAP_B), ("MAP_C", MAP_C)):
            run_map(n, g, table)
    print_table(table)

MAP_A   start=(0, 0)  goal=(4, 7)

--- UCS ---
Path           : [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Path length    : 11 moves
Total cost     : 15
Nodes expanded : 28
Map (* = path):
S**#....
##*#.##.
..******
.####.#*
...~..#G

--- A* (Manhattan) ---
Path           : [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (3, 7), (4, 7)]
Path length    : 11 moves
Total cost     : 15
Nodes expanded : 21
Map (* = path):
S**#....
##*#.##.
..******
.####.#*
...~..#G

MAP_B   start=(0, 0)  goal=(0, 8)

--- UCS ---
Path           : [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (1, 8), (0, 8)]
Path length    : 12 moves
Total cost     : 12
Nodes expanded : 30
Map (* = path):
S~~~~~~~G
*#######*
*********
.#.#.#.#.
.........

--- A* (Manhattan) ---
Path           : [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (1, 8), (0, 8)]